# Ocean Waves (HRES-WAM) — flash-aurora Engine

> **Personal users:** This tutorial is **not runnable end-to-end** with a self-registered ECMWF account. Wave initial conditions come from the **MARS archive** (`stream=wave`, `type=an`), which requires **institutional MARS authorisation** — an API key alone is not enough. See [ECMWF MARS FAQ](https://confluence.ecmwf.int/display/UDOC/ecmwf.API+error+1%3A+User+has+no+access+to+services+mars+-+Web+API+FAQ). Use another example notebook (e.g. `example_era5.ipynb`, `example_hres_t0.ipynb`) for personal testing.

Reference setup matching the upstream [Microsoft Aurora Wave example](microsoft-aurora/docs/example_wave.ipynb) — for teams with **MARS access** or **pre-staged** tutorial GRIB:

- **Date:** 2022-09-16
- **Initial condition:** 06:00 UTC (00:00 + 06:00 history for met; same for WAM)
- **Rollout:** 2 steps → 12:00 and 18:00 UTC
- **Model:** `AuroraWave` via preset `wave`

Inputs combine **HRES-WAM** wave fields (MARS) and **HRES T0** meteorology (WeatherBench2). Met files can be fetched from WeatherBench2; wave GRIB must come from MARS or a pre-staged copy under `<ASSET_ROOT>/wave/`.

## Prerequisites

1. **Extra packages:** `uv pip install gcsfs zarr cfgrib matplotlib` (`ecmwf-api-client` only if you have MARS access).
2. **Wave GRIB:** `{day}-wave.grib` at `<ASSET_ROOT>/wave/` — **not** auto-downloaded for personal users. Institutional users: MARS or Computing Representative access ([archive datasets](https://www.ecmwf.int/en/forecasts/access-forecasts/access-archive-datasets)).
3. **Checkpoint** and **`aurora-0.25-wave-static.pickle`** under `ASSET_ROOT`, or Hugging Face download in the setup cell.


In [ ]:
from datetime import datetime
from pathlib import Path

from flash_aurora.engine import (
    DEFAULT_PRESETS,
    DataDownloader,
    HF_MIRROR_ENDPOINT,
)
from flash_aurora.engine.core.redaction import safe_path

PRESET = "wave"
VALID_TIME = datetime(2022, 9, 16, 6)
TIME_INDEX = 1
ROLLOUT_STEPS = 2

# Named tier or combo: backbone@encoder_decoder (see README).
INFERENCE_PRECISION = "bf16_mixed@fp32"

# Default: ./assets under the notebook working directory (created if missing).
ASSET_ROOT: Path | str | None = None

# Optional — absolute path to a mounted data disk with checkpoints/cache (uncomment to use):
ASSET_ROOT = Path("/root/autodl-tmp/aurora")

if ASSET_ROOT is not None:
    root = Path(ASSET_ROOT).expanduser()
    if not root.is_absolute():
        raise ValueError("ASSET_ROOT must be an absolute path")
    ASSET_ROOT = root.resolve()
else:
    ASSET_ROOT = (Path.cwd() / "assets").resolve()
    ASSET_ROOT.mkdir(parents=True, exist_ok=True)

variant = DEFAULT_PRESETS.get(PRESET).variant
CHECKPOINT_PATH = ASSET_ROOT / variant.checkpoint_filename
USE_HF_MIRROR = False  # True -> https://hf-mirror.com when huggingface.co is blocked

if CHECKPOINT_PATH.is_file():
    CHECKPOINT_ARG = CHECKPOINT_PATH
    ALLOW_HUB_DOWNLOAD = False
    HF_ENDPOINT = None
    print("checkpoint: local", safe_path(CHECKPOINT_PATH))
else:
    CHECKPOINT_ARG = None
    ALLOW_HUB_DOWNLOAD = True
    HF_ENDPOINT = HF_MIRROR_ENDPOINT if USE_HF_MIRROR else None
    print("checkpoint: missing locally; will download from Hugging Face")
    print("  target dir:", safe_path(ASSET_ROOT))
    print("  filename:", variant.checkpoint_filename)
    print("  hf_endpoint:", HF_ENDPOINT or "https://huggingface.co (default)")

downloader = DataDownloader.from_preset(PRESET, asset_root=ASSET_ROOT)
cache_dir = downloader.resolve_cache_dir()

print("cache_dir:", safe_path(cache_dir))
print("asset_root:", safe_path(ASSET_ROOT))
print("allow_hub_download:", ALLOW_HUB_DOWNLOAD)

from flash_aurora.engine.core.hub import HubDownloadOptions
from flash_aurora.engine.core.paths import AssetStore

# Static fields from Hugging Face (file lives in ASSET_ROOT, not under the ingress cache).
STATIC_PICKLE_PATH = ASSET_ROOT / variant.static_pickle
HF_OPTIONS = HubDownloadOptions(
    endpoint=HF_MIRROR_ENDPOINT if USE_HF_MIRROR else HF_ENDPOINT,
)
if STATIC_PICKLE_PATH.is_file():
    print("static_pickle: local", safe_path(STATIC_PICKLE_PATH))
else:
    print("static_pickle: missing locally; will download from Hugging Face")
    print("  target dir:", safe_path(ASSET_ROOT))
    print("  filename:", variant.static_pickle)
    print("  hf_endpoint:", HF_OPTIONS.endpoint or "https://huggingface.co (default)")
    STATIC_PICKLE_PATH = AssetStore(root=ASSET_ROOT).fetch_hub_file(
        variant.static_pickle,
        repo=variant.hf_repo,
        allow_download=True,
        explicit=ASSET_ROOT,
        hub=HF_OPTIONS,
    )
    print("static_pickle: ready", safe_path(STATIC_PICKLE_PATH))


## 1. Cache check (MARS not used for personal accounts)

`DataDownloader` can fetch HRES T0 NetCDF from WeatherBench2, but **`{day}-wave.grib` requires MARS** (or a file you place manually). The next cell stops if wave data is missing — personal users should **stop here** and use another tutorial.

| File | Source | Personal user |
|------|--------|---------------|
| `{day}-wave.grib` | ECMWF MARS archive | ❌ not available |
| `{day}-surface-level.nc`, `{day}-atmospheric.nc` | WeatherBench2 | ✅ public |


In [ ]:
from flash_aurora.engine import DownloadResult
from flash_aurora.engine.core.redaction import safe_path

WAVE_DAY = VALID_TIME.strftime("%Y-%m-%d")
wave_grib = cache_dir / f"{WAVE_DAY}-wave.grib"
missing = downloader.missing(VALID_TIME)

print("Wave IC requires ECMWF MARS archive access (not available to personal/self-registered accounts).")
print("Expected:", safe_path(wave_grib))
print("Missing keys:", missing)

if "wave" in missing:
    raise SystemExit(
        "Skip this tutorial: obtain 2022-09-16-wave.grib from an authorised MARS user, "
        "or run example_era5 / example_hres_t0 instead."
    )

if missing:
    result = downloader.ensure(VALID_TIME)
else:
    print("Wave cache complete under", safe_path(cache_dir))
    result = DownloadResult(
        cache_dir=cache_dir,
        paths=downloader.expected_paths(VALID_TIME),
        downloaded=(),
        skipped=tuple(downloader.expected_paths(VALID_TIME)),
    )

print("downloaded:", result.downloaded)
print("skipped:", result.skipped)
for key, path in result.paths.items():
    print(f"  {key}: {safe_path(path)}")


## 1. Download HRES-WAM + HRES T0

`downloader.ensure()` writes:

| File | Source |
|------|--------|
| `{day}-wave.grib` | ECMWF MARS (`stream=wave`) |
| `{day}-surface-level.nc`, `{day}-atmospheric.nc` | WeatherBench2 HRES T0 |

Skip network calls when files already exist on the data disk.


In [ ]:
from flash_aurora.engine import DownloadResult
from flash_aurora.engine.core.redaction import safe_path
from flash_aurora.engine.ingress.download.credentials import merge_credentials

missing = downloader.missing(VALID_TIME)

if missing:
    ready = merge_credentials().ecmwf_settings() is not None
    print("Missing cache files:", missing)
    print("ECMWF credentials ready:", ready)
    result = downloader.ensure(VALID_TIME, prompt=not ready)
else:
    print("Wave cache already complete under", safe_path(cache_dir))
    result = DownloadResult(
        cache_dir=cache_dir,
        paths=downloader.expected_paths(VALID_TIME),
        downloaded=(),
        skipped=tuple(downloader.expected_paths(VALID_TIME)),
    )

print("downloaded:", result.downloaded)
print("skipped:", result.skipped)
for key, path in result.paths.items():
    print(f"  {key}: {safe_path(path)}")


## 2–3. Build IC, load, rollout

Requires a complete cache (including wave GRIB). **Stop above if you are a personal user without MARS.**

`Wb2WamWaveAdapter` merges flipped HRES T0 met fields with WAM wave variables. Post-processing for near-zero wave height is applied at plot time (upstream Supplementary Information §C.5).


In [ ]:
import torch

from flash_aurora.engine import AuroraEngine, InitialConditionBuilder
from flash_aurora.aurora.model.inference_precision import describe_inference_config

engine = AuroraEngine.from_preset(
    PRESET,
    asset_root=ASSET_ROOT,
    checkpoint_path=CHECKPOINT_ARG,
    allow_hub_download=ALLOW_HUB_DOWNLOAD,
    hf_mirror=USE_HF_MIRROR,
    hf_endpoint=None if USE_HF_MIRROR else HF_ENDPOINT,
)

# Must be set before load() — see README "Inference precision tiers".
engine.config.inference_precision = INFERENCE_PRECISION
engine.config.gpu_rollout_steps = ROLLOUT_STEPS

request = downloader.ingest_request(
    VALID_TIME,
    time_index=TIME_INDEX,
    download=False,
)
builder = InitialConditionBuilder(engine.config)
batch = builder.from_source(request)
print("IC time:", batch.metadata.time)
print("spatial:", batch.spatial_shape)

engine.load()
print("device:", next(engine.model.parameters()).device)

cfg = engine.model.inference_config
if cfg is not None:
    print("inference tier:", cfg.config_label)
    print(describe_inference_config(cfg))

preds = []
with torch.inference_mode():
    for pred in engine.rollout_stream(batch, ROLLOUT_STEPS):
        preds.append(pred.to("cpu"))
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print("predictions:", [str(p.metadata.time[0]) for p in preds])
engine.release_gpu()


## 4. Visualize mean wave direction (MWD)

Left: Aurora forecast. Right: HRES-WAM reference. Directions masked when SWH < 1e-4 m.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

wave_vars_ds = xr.open_dataset(
    result.paths["wave"],
    engine="cfgrib",
    backend_kwargs={"indexpath": ""},
)

fig, axs = plt.subplots(2, 2, figsize=(12, 6.5))

for i in range(axs.shape[0]):
    pred = preds[i]

    ax = axs[i, 0]
    ax.imshow(pred.surf_vars["mwd"][0, 0].numpy(), vmin=0, vmax=360, cmap="twilight")
    ax.set_ylabel(str(pred.metadata.time[0]))
    if i == 0:
        ax.set_title("Aurora Prediction (flash-aurora Engine)")
    ax.set_xticks([])
    ax.set_yticks([])

    ax = axs[i, 1]
    ref = wave_vars_ds["mwd"][2 + i].values
    ref[wave_vars_ds["swh"][2 + i].values < 1e-4] = np.nan
    ax.imshow(ref, vmin=0, vmax=360, cmap="twilight")
    if i == 0:
        ax.set_title("HRES-WAM")
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.show()


## 5. (Optional) Export rollout to NetCDF


In [ ]:
# Model already released in section 3.
print("model device:", next(engine.model.parameters()).device)
print("gpu ticket:", engine._gpu_ticket)

# Re-allocate model to GPU
engine.model.to(engine.config.device)
import torch
torch.cuda.synchronize()
print("model device:", next(engine.model.parameters()).device)

# Export rollout to NetCDF
EXPORT_DIR = ASSET_ROOT / "output" / PRESET
paths = list(engine.rollout_and_export(batch, steps=ROLLOUT_STEPS, export_dir=EXPORT_DIR))
for p in paths:
    print(safe_path(p), p.is_file())
    
engine.release_gpu()
